In [1]:
%load_ext autoreload
%autoreload 2
import moabb
from moabb.datasets import *
from moabb.paradigms import P300
from hoda import HODA
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from mne.viz import plot_topomap
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import roc_auc_score
from sklearn.manifold import TSNE
from mne.decoding import Scaler
from mne import combine_evoked
from mda import ParafacMDA

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


2022-08-25 09:46:09.314643: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2022-08-25 09:46:09.314667: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [2]:
paradigm = P300(resample=32, tmin=0, tmax=1, fmin=0.5, fmax=16)
dataset = BNCI2014008()

In [3]:
subject, session, run = 2, "session_0", "run_0"
dataset.download(subject_list=[subject])
data = dataset.get_data(subjects=[subject])
raw = data[subject][session][run]
raw.info
epochs, labels, _ = paradigm.process_raw(raw, dataset, return_epochs=True)
epochs

Creating RawArray with float64 data, n_channels=10, n_times=347704
    Range : 0 ... 347703 =      0.000 ...  1358.215 secs
Ready.
Not setting metadata
4200 matching events found
No baseline correction applied
0 bad epochs dropped


Number of events,4200
Events,NonTarget: 3500Target: 700
Time range,0.000 – 0.969 sec
Baseline,off


In [4]:
contrast = combine_evoked([epochs['Target'].average(), epochs['NonTarget'].average()], weights=[1,-1])

In [5]:
#import scipy.signal
#X = scipy.signal.hilbert(epochs.get_data()).astype(np.float64)
X = epochs.get_data().astype(np.float64)
y = labels
print(X.shape)

(4200, 8, 32)


In [6]:
X -= np.mean(X, axis=(0,2))[np.newaxis,:,np.newaxis]
X /= np.std(X, axis=(0,2))[np.newaxis,:,np.newaxis]

In [8]:
rank = 2
hoda = ParafacMDA(rank=rank)
hoda.fit(X,y)

Optimizing...
Iteration    Cost                       Gradient norm     
---------    -----------------------    --------------    
2.4867014972887917e-05


NotImplementedError: No autodiff support available for the canonical 'NumPy' backend

In [ ]:
for k in range(2):
    vmax=np.max(np.abs(hoda.scatter_w_[k]))
    sns.heatmap(hoda.scatter_w_[k], cmap='BrBG', vmin=-vmax, vmax=vmax)
    plt.show()

In [ ]:
for k in range(2):
    vmax=np.max(np.abs(hoda.scatter_b_[k]))
    sns.heatmap(hoda.scatter_b_[k], cmap='BrBG', vmin=-vmax, vmax=vmax)
    plt.show()

In [ ]:
fig, axs = plt.subplots(2,3, figsize=(16,6*2))
for r in range(rank):
    plot_topomap(hoda.proj_.factors[0][:,r] @ hoda.scatter_w_[0], epochs.info, axes=axs[0,r], show=False)
    axs[1,r].plot(epochs.times,hoda.proj_.factors[1][:,r] @ hoda.scatter_w_[1])
    axs[1,r].set_xlabel('Time (s)')
    axs[1,r].set_ylabel('Weight')

In [ ]:
Xt = hoda.transform(X)

In [ ]:
pca = PCA(n_components=2)
X_pc = pca.fit_transform(Xt)
plt.scatter(X_pc[:,0], X_pc[:,1],c=labels=='Target')

In [ ]:
lda = LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
lda.fit(Xt,y)

In [ ]:
roc_auc_score(y, lda.predict_proba(Xt)[:,1])